## Finetune Llama-2-7b

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model_name = "meta-llama/Llama-2-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # 避免 padding 报错
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model, tokenizer


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 19,988,480 || all params: 6,758,404,096 || trainable%: 0.2957573965106688


(PeftModelForCausalLM(
   (base_model): LoraModel(
     (model): LlamaForCausalLM(
       (model): LlamaModel(
         (embed_tokens): Embedding(32000, 4096)
         (layers): ModuleList(
           (0-31): 32 x LlamaDecoderLayer(
             (self_attn): LlamaAttention(
               (q_proj): lora.Linear4bit(
                 (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                 (lora_dropout): ModuleDict(
                   (default): Identity()
                 )
                 (lora_A): ModuleDict(
                   (default): Linear(in_features=4096, out_features=8, bias=False)
                 )
                 (lora_B): ModuleDict(
                   (default): Linear(in_features=8, out_features=4096, bias=False)
                 )
                 (lora_embedding_A): ParameterDict()
                 (lora_embedding_B): ParameterDict()
               )
               (k_proj): lora.Linear4bit(
                 (base_layer): Linear4bi

In [20]:
tokenizer("hello world", return_tensors="pt")

{'input_ids': tensor([[    1, 22172,  3186]]), 'attention_mask': tensor([[1, 1, 1]])}

In [14]:
from datasets import load_dataset

def formatting(example):
    tokens = tokenizer(
        f"### instruction: {example['instruction']}\n\n### input: {example['input']}\n\n### output: {example['output']}",
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = load_dataset("json", data_files="./data/datasets/mikewei/myinfo.json")
tokenized_dataset = dataset["train"].map(formatting) # type: ignore

training_args = TrainingArguments(
    output_dir="./testing/llama-ft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=30,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,  # 4090 支持
    logging_dir="./testing/llama-ft",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipykernel_9413/1451168529.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/home/mikewei/anaconda3/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.349400
20,0.090100
30,0.012100


TrainOutput(global_step=30, training_loss=0.4838836786647638, metrics={'train_runtime': 62.322, 'train_samples_per_second': 2.888, 'train_steps_per_second': 0.481, 'total_flos': 3664649555804160.0, 'train_loss': 0.4838836786647638, 'epoch': 30.0})

In [19]:
import torch

prompt = """### instruction: 谁是mikewei?
### input:
### output:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 生成（推理）
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.95,
        temperature=0.7
    )

# 解码结果
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("模型输出：\n", response)

模型输出：
 ### instruction: 谁是mikewei?
### input:
### output: mikewei是一名高端技术专家，擅长产出高效率、稳定、可靠的系统解决方案。
